In [40]:
# Librerias
import pandas as pd
import numpy as np
import matplotlib.pyplot as pl
import seaborn as sn

In [41]:
# Leer Dataset
df = pd.read_csv("../data/raw/dirty_cafe_sales.csv")

In [42]:
df.head()

,Transaction ID,Item,Quantity,Price Per Unit,Total Spent,Payment Method,Location,Transaction Date
0,TXN_1961373,Coffee,2,2.0,4.0,Credit Card,Takeaway,2023-09-08
1,TXN_4977031,Cake,4,3.0,12.0,Cash,In-store,2023-05-16
2,TXN_4271903,Cookie,4,1.0,ERROR,Credit Card,In-store,2023-07-19
3,TXN_7034554,Salad,2,5.0,10.0,UNKNOWN,UNKNOWN,2023-04-27
4,TXN_3160411,Coffee,2,2.0,4.0,Digital Wallet,In-store,2023-06-11


In [43]:
df.info()

<class 'pandas.DataFrame'>
RangeIndex: 10000 entries, 0 to 9999
Data columns (total 8 columns):
 #   Column            Non-Null Count  Dtype
---  ------            --------------  -----
 0   Transaction ID    10000 non-null  str  
 1   Item              9667 non-null   str  
 2   Quantity          9862 non-null   str  
 3   Price Per Unit    9821 non-null   str  
 4   Total Spent       9827 non-null   str  
 5   Payment Method    7421 non-null   str  
 6   Location          6735 non-null   str  
 7   Transaction Date  9841 non-null   str  
dtypes: str(8)
memory usage: 625.1 KB


In [44]:
df.describe()

,Transaction ID,Item,Quantity,Price Per Unit,Total Spent,Payment Method,Location,Transaction Date
count,10000,9667,9862,9821,9827,7421,6735,9841
unique,10000,10,7,8,19,5,4,367
top,TXN_1961373,Juice,5,3.0,6.0,Digital Wallet,Takeaway,UNKNOWN
freq,1,1171,2013,2429,979,2291,3022,159


In [45]:
# Conversión de tipos de datos
# Se convierten columnas numéricas a formato numérico
# fecha a datetime para análisis

cols_num = ["Quantity", "Price Per Unit", "Total Spent"]
for col in cols_num:
    df[col] = pd.to_numeric(df[col], errors="coerce")

df["Transaction Date"] = pd.to_datetime(df["Transaction Date"], errors="coerce")

In [46]:
df.info()

<class 'pandas.DataFrame'>
RangeIndex: 10000 entries, 0 to 9999
Data columns (total 8 columns):
 #   Column            Non-Null Count  Dtype         
---  ------            --------------  -----         
 0   Transaction ID    10000 non-null  str           
 1   Item              9667 non-null   str           
 2   Quantity          9521 non-null   float64       
 3   Price Per Unit    9467 non-null   float64       
 4   Total Spent       9498 non-null   float64       
 5   Payment Method    7421 non-null   str           
 6   Location          6735 non-null   str           
 7   Transaction Date  9540 non-null   datetime64[us]
dtypes: datetime64[us](1), float64(3), str(4)
memory usage: 625.1 KB


In [47]:
# Revisar valores nulos
df.isnull().sum()

Transaction ID         0
Item                 333
Quantity             479
Price Per Unit       533
Total Spent          502
Payment Method      2579
Location            3265
Transaction Date     460
dtype: int64

In [48]:
missing = df.isnull().mean()*100
missing.sort_values(ascending=False)

Location            32.65
Payment Method      25.79
Price Per Unit       5.33
Total Spent          5.02
Quantity             4.79
Transaction Date     4.60
Item                 3.33
Transaction ID       0.00
dtype: float64

In [49]:
# Exploración de variables categóricas
# Revisación de valores más frecuentes en columnas de tipo texto
# para detectar inconsistencias

cols_text = ["Item", "Payment Method", "Location"]
for col in cols_text:
    print(f"\n-------")
    print(df[col].value_counts(dropna=False).head(10))


-------
Item
Juice       1171
Coffee      1165
Salad       1148
Cake        1139
Sandwich    1131
Smoothie    1096
Cookie      1092
Tea         1089
UNKNOWN      344
NaN          333
Name: count, dtype: int64

-------
Payment Method
NaN               2579
Digital Wallet    2291
Credit Card       2273
Cash              2258
ERROR              306
UNKNOWN            293
Name: count, dtype: int64

-------
Location
NaN         3265
Takeaway    3022
In-store    3017
ERROR        358
UNKNOWN      338
Name: count, dtype: int64


In [50]:
# Reemplazar valores de error y etiquetas de datos faltantes ("ERROR", "UNKNOWN")
# Por NaN para unificar el tratamiento de los datos faltantes
df.replace(["ERROR", "UNKNOWN"], pd.NA, inplace=True)
missing = df.isnull().mean()*100
missing.sort_values(ascending=False)

Location            39.61
Payment Method      31.78
Item                 9.69
Price Per Unit       5.33
Total Spent          5.02
Quantity             4.79
Transaction Date     4.60
Transaction ID       0.00
dtype: float64

In [51]:
cols_fill = ["Location", "Payment Method"]
for col in cols_fill:
    df[col] = df[col].fillna("Unknown")
missing = df.isnull().mean()*100
missing.sort_values(ascending=False)

Item                9.69
Price Per Unit      5.33
Total Spent         5.02
Quantity            4.79
Transaction Date    4.60
Transaction ID      0.00
Payment Method      0.00
Location            0.00
dtype: float64

In [52]:
# Imputación de valores faltantes en "Price Per Unit" usando la mediana por producto (Item)
# Se ve que cada producto tiene un rango de precios relativamente estable,
# por lo que la mediana por Item se utiliza como estimación robusta frente a outliers

price_map = df.groupby("Item")["Price Per Unit"].median().to_dict()
df["Price Per Unit"] = df["Price Per Unit"].fillna(df["Item"].map(price_map))

In [53]:
# Imputación de valores faltantes en "Total Spent", "Quantity", "Price Per Unit", cuando es posible calcularlos
mask_total = (
    df["Total Spent"].isna() &
    df["Quantity"].notna() &
    df["Price Per Unit"].notna()
)

df.loc[mask_total, "Total Spent"] = (
    df.loc[mask_total, "Quantity"] * df.loc[mask_total, "Price Per Unit"]
)

mask_quantity = (
    df["Quantity"].isna() &
    df["Total Spent"].notna() &
    df["Price Per Unit"].notna()
)

df.loc[mask_quantity, "Quantity"] = (
    df.loc[mask_quantity, "Total Spent"] / df.loc[mask_quantity, "Price Per Unit"]
)

mask_price = (
    df["Price Per Unit"].isna() &
    df["Quantity"].notna() &
    df["Total Spent"].notna()
)

df.loc[mask_price, "Price Per Unit"] = (
    df.loc[mask_price, "Total Spent"] / df.loc[mask_price, "Quantity"]
)

In [54]:
missing = df.isnull().mean()*100
missing.sort_values(ascending=False)

Item                9.69
Transaction Date    4.60
Quantity            0.23
Total Spent         0.23
Price Per Unit      0.06
Transaction ID      0.00
Payment Method      0.00
Location            0.00
dtype: float64

In [55]:
# Se elimina filas con valores faltantes en Item y Transaction Date, porque no es posible analizar ventas sin conocer el producto
df = df.dropna(subset=["Item", "Transaction Date"])

In [56]:
# Se elimina filas con valores faltantes en Quantity y Total Spent
df = df.dropna(subset=["Quantity", "Total Spent"])
missing = df.isnull().mean()*100
missing.sort_values(ascending=False)

Transaction ID      0.0
Item                0.0
Quantity            0.0
Price Per Unit      0.0
Total Spent         0.0
Payment Method      0.0
Location            0.0
Transaction Date    0.0
dtype: float64

In [58]:
# Nueva columna fecha separada por mes y día de la semana
df["Month"] = df["Transaction Date"].dt.month_name()
df["Day"] = df["Transaction Date"].dt.day_name()

In [59]:
df.to_csv("../data/processed/coffee_sales_clean.csv", index=False) 

In [60]:
df.info()

<class 'pandas.DataFrame'>
Index: 8593 entries, 0 to 9999
Data columns (total 10 columns):
 #   Column            Non-Null Count  Dtype         
---  ------            --------------  -----         
 0   Transaction ID    8593 non-null   str           
 1   Item              8593 non-null   str           
 2   Quantity          8593 non-null   float64       
 3   Price Per Unit    8593 non-null   float64       
 4   Total Spent       8593 non-null   float64       
 5   Payment Method    8593 non-null   str           
 6   Location          8593 non-null   str           
 7   Transaction Date  8593 non-null   datetime64[us]
 8   Month             8593 non-null   str           
 9   Day               8593 non-null   str           
dtypes: datetime64[us](1), float64(3), str(6)
memory usage: 738.5 KB
